# Train Qwen3 1.7B LoRA Adapters

Train one adapter on theorem-explicit reasoning and one adapter on theorem-implicit reasoning.

In [ ]:
!pip install -q -U "mlx-lm[train]" pandas tqdm

In [ ]:
from pathlib import Path
import shlex
import subprocess
import sys

In [ ]:
PROJECT_ROOT = Path("/Users/xingzheli/Documents/Python-WorkSpace/SC_fine-tune")
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B-MLX-bf16"
DATA_ROOT = PROJECT_ROOT / "training_eval" / "fine_tune_qwen1_7B" / "lora" / "data"
RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora"

In [ ]:
ITERS = 80
BATCH_SIZE = 1
GRAD_ACCUMULATION_STEPS = 8
NUM_LAYERS = 4
LEARNING_RATE = "1e-5"
SKIP_IF_ADAPTER_EXISTS = True

In [ ]:
EXPERIMENTS = {
    "explicit_theorems": {
        "data_dir": DATA_ROOT / "explicit_theorems",
        "adapter_path": RESULT_ROOT / "explicit_theorems" / "adapters",
    },
    "implicit_theorems": {
        "data_dir": DATA_ROOT / "implicit_theorems",
        "adapter_path": RESULT_ROOT / "implicit_theorems" / "adapters",
    },
}

In [ ]:
def make_lora_command(data_dir, adapter_path):
    return [
        sys.executable,
        "-m",
        "mlx_lm.lora",
        "--model",
        MODEL_NAME,
        "--train",
        "--data",
        str(data_dir),
        "--adapter-path",
        str(adapter_path),
        "--iters",
        str(ITERS),
        "--batch-size",
        str(BATCH_SIZE),
        "--grad-accumulation-steps",
        str(GRAD_ACCUMULATION_STEPS),
        "--num-layers",
        str(NUM_LAYERS),
        "--learning-rate",
        LEARNING_RATE,
        "--mask-prompt",
        "--grad-checkpoint",
    ]

In [ ]:
def adapter_exists(adapter_path):
    return (adapter_path / "adapters.safetensors").exists() or (adapter_path / "adapters.npz").exists()

In [ ]:
def run_lora_training(name, config):
    if SKIP_IF_ADAPTER_EXISTS and adapter_exists(config["adapter_path"]):
        print(f"Skipping {name}; adapter already exists at {config['adapter_path']}")
        return

    config["adapter_path"].mkdir(parents=True, exist_ok=True)
    command = make_lora_command(config["data_dir"], config["adapter_path"])
    print(shlex.join(command))
    subprocess.run(command, check=True)

## Smoke Train

Run this first. It trains only the explicit adapter with the current short `ITERS` setting. If this works, run both adapters below.

In [ ]:
RUN_SMOKE = False

if RUN_SMOKE:
    run_lora_training("explicit_theorems", EXPERIMENTS["explicit_theorems"])

## Full Pair

After the smoke train is fine, increase `ITERS` if desired, restart the notebook, and run this cell.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    for name, config in EXPERIMENTS.items():
        run_lora_training(name, config)

## Validation Accuracy

Use these cells after training to evaluate the adapters on validation data. This is the right split for choosing LoRA hyperparameters. The frozen `benchmark/data/test` split should be used only after the setup is chosen.

Validation is closed-book: the prompt contains only the problem, not the training reasoning. Metrics are reported separately for binary true/false-style tasks and non-binary short-answer tasks.

In [ ]:
from mlx_lm import generate, load
from tqdm.auto import tqdm

In [ ]:
from training_eval.eval_utils import (
    extract_json_object,
    is_correct,
    load_jsonl_records,
    make_fine_tuned_chat_messages,
    rows_to_frame,
    save_results,
    summarize_accuracy,
)

In [ ]:
VAL_RECORDS = {
    "explicit_theorems": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val"),
    "implicit_theorems": load_jsonl_records(PROJECT_ROOT / "benchmark" / "data" / "val_implicit_theorems"),
}

VALIDATION_RESULT_ROOT = PROJECT_ROOT / "results" / "fine_tunes" / "qwen3_1_7b_lora_validation"
MAX_NEW_TOKENS = 256

In [ ]:
def make_qwen_prompt(tokenizer, problem):
    messages = make_fine_tuned_chat_messages(problem)
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

In [ ]:
def load_adapter_model(adapter_path):
    return load(MODEL_NAME, adapter_path=str(adapter_path))

In [ ]:
def generate_validation_answer(model, tokenizer, problem):
    prompt = make_qwen_prompt(tokenizer, problem)
    return generate(model, tokenizer, prompt=prompt, max_tokens=MAX_NEW_TOKENS, verbose=False)

In [ ]:
def evaluate_adapter_on_validation(adapter_label, config):
    model, tokenizer = load_adapter_model(config["adapter_path"])
    rows = []

    for record in tqdm(VAL_RECORDS[adapter_label], desc=f"val/{adapter_label}"):
        raw_output = generate_validation_answer(model, tokenizer, record["problem"])
        predicted = extract_json_object(raw_output)
        metadata = record.get("metadata", {})

        rows.append({
            "adapter": adapter_label,
            "id": record["id"],
            "family": record["family"],
            "problem_type": record["problem_type"],
            "difficulty": record["difficulty"],
            "answer_type": record["answer_type"],
            "manual_variation": metadata.get("manual_variation", False),
            "problem": record["problem"],
            "canonical_answer": record["canonical_answer"],
            "raw_output": raw_output,
            "predicted_answer": predicted,
            "correct": is_correct(predicted, record["canonical_answer"]),
        })

    return rows

In [ ]:
def save_validation_results(adapter_label, rows):
    df = rows_to_frame(rows)
    metrics = summarize_accuracy(df)
    metrics.update({
        "model": MODEL_NAME,
        "adapter": adapter_label,
        "dataset": "benchmark/data/val or benchmark/data/val_implicit_theorems",
        "split_role": "validation_for_hyperparameter_selection",
    })
    return save_results(rows, VALIDATION_RESULT_ROOT / adapter_label, metrics)

In [ ]:
validation_rows = []

for adapter_label, config in EXPERIMENTS.items():
    rows = evaluate_adapter_on_validation(adapter_label, config)
    save_validation_results(adapter_label, rows)
    validation_rows.extend(rows)

val_df = rows_to_frame(validation_rows)
val_df.head()

In [ ]:
display(val_df.groupby("adapter")["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "answer_type"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "family"])["correct"].agg(["mean", "sum", "count"]).sort_index())
display(val_df.groupby(["adapter", "difficulty"])["correct"].agg(["mean", "sum", "count"]).sort_index())

In [ ]:
val_df.loc[
    ~val_df["correct"],
    ["adapter", "id", "answer_type", "family", "problem_type", "difficulty", "canonical_answer", "predicted_answer", "raw_output"],
].head(30)